# 01B — Common Canonical Adapter

**Outcome:** convert any valid sector pack into the same canonical
`SPEC-CORE` and optional, physically separate `SPEC-EVAL`.

This notebook does not know what an ONT, splitter, oil well, valve, FEC count,
or 3W class means. It uses only the pack manifest and the standard pack table
schemas.


## 1. Setup

Run at least one `01A` sector-pack notebook first. The first code cell mounts
Drive and imports the shared adapter. The following **sector switch** cell is
the only block you edit when moving between Telecom and Petrobras 3W.


In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    EVAL_SCHEMAS,
    audit_core,
    core_content_hashes,
    finalise_pack,
    materialise_canonical,
    read_json,
    runtime_probe,
    validate_pack,
    write_json,
)


### Choose the sector here

Change only `SECTOR` when switching between the two packs. The dictionary
contains output folder names, not translation logic. The adapter below still
uses only the standardized pack interface.

For a new materialisation leave `BUILD_CANONICAL = True`. To inspect an
existing completed run without rebuilding it, set `BUILD_CANONICAL = False`.
Successful output directories are immutable.


In [ ]:
# Change only this value: "telecom" or "petrobras_3w"
SECTOR = "telecom"
BUILD_CANONICAL = True  # False = inspect an existing run only
SECTOR = os.getenv("ADAPTER_SECTOR", SECTOR)

PACK_RUN_IDS = {
    "telecom": "telecom_v4_1_full_v1",
    "petrobras_3w": "real_well_contract_fixture_v1",
}
if SECTOR not in PACK_RUN_IDS:
    raise ValueError(
        f"Unknown sector {SECTOR!r}. Choose one of {list(PACK_RUN_IDS)}"
    )

PACK_RUN_ID = os.getenv(
    "ADAPTER_PACK_RUN_ID",
    PACK_RUN_IDS[SECTOR],
)
CANONICAL_RUN_ID = os.getenv(
    "CANONICAL_RUN_ID",
    f"{SECTOR}_canonical_v1",
)
PACK_ROOT = Path(os.getenv(
    "ADAPTER_PACK_ROOT",
    DRIVE_ROOT / "outputs" / "packs" / SECTOR / PACK_RUN_ID,
))
RUN_BUILD = os.getenv(
    "RUN_CANONICAL_ADAPTER",
    "1" if BUILD_CANONICAL else "0",
) == "1"

pack_manifest = validate_pack(PACK_ROOT)
if pack_manifest["sector"] != SECTOR:
    raise ValueError(
        f"Selected {SECTOR!r}, but the pack contains "
        f"{pack_manifest['sector']!r}"
    )

RUN_ROOT = (
    DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CANONICAL_RUN_ID
)

display(pd.Series({
    "selected_sector": SECTOR,
    "pack_root": str(PACK_ROOT),
    "canonical_run_root": str(RUN_ROOT),
    "contract_version": CORE_VERSION,
}, name="value").to_frame())


## 2. The neutral contract

`SPEC-CORE` is the only input later given to EDA, feature engineering, and
models. `SPEC-EVAL` is opened only after predictions are frozen.

The pack interface is deliberately slightly different from the canonical
contract: observations remain wide for storage efficiency; this adapter turns
them into the long telemetry table used by common analysis code.


In [ ]:
contract_rows = [
    {
        "box": "SPEC-CORE",
        "table": table,
        "fields": ", ".join(columns),
    }
    for table, columns in CORE_SCHEMAS.items()
]
contract_rows += [
    {
        "box": "SPEC-EVAL",
        "table": table,
        "fields": ", ".join(EVAL_SCHEMAS[table]),
    }
    for table in pack_manifest["evaluation_tables"]
]
display(pd.DataFrame(contract_rows))

pack_summary = {
    "sector": pack_manifest["sector"],
    "pack_version": pack_manifest["pack_version"],
    "cadence_seconds": pack_manifest["expected_cadence_seconds"],
    "metrics": len(pack_manifest["metric_ids"]),
    "evaluation_tables": pack_manifest["evaluation_tables"],
}
display(pd.Series(pack_summary, name="value").to_frame())


## 3. Materialise canonical data

The adapter performs only five operations:

1. validate the pack interface and content hashes;
2. reshape wide observations to long telemetry in physical batches;
3. mark source nulls `invalid` and values at declared ceilings `clipped`;
4. derive collection gaps from cadence and entity validity;
5. copy evaluation tables into a separate directory.

It does not create lags, rolling statistics, anomaly scores, exposure
denominators, or sector-specific interpretations.


In [ ]:
if RUN_BUILD:
    workflow = materialise_canonical(
        PACK_ROOT,
        RUN_ROOT,
        include_evaluation=True,
    )
else:
    workflow = read_json(RUN_ROOT / "workflow_report.json")

CORE = RUN_ROOT / "SPEC-CORE"
EVALUATION = RUN_ROOT / "SPEC-EVAL"
audit = audit_core(CORE)

display(pd.Series(audit, name="value").to_frame())
display(pd.DataFrame({
    "metric_id": pack_manifest["metric_ids"],
}))


## 4. Common-adapter truth-isolation test

The sector-pack notebooks contain the primary leakage proof because they see
native truth. This second test verifies the common boundary:

- create a small pack with `PACK-EVAL` mounted;
- create the same `PACK-CORE` with `PACK-EVAL` absent;
- run the common adapter on both;
- compare canonical `SPEC-CORE` logical hashes.

The runtime probe is deployment evidence, not the primary leakage proof.


In [ ]:
def make_adapter_fixture(source_pack, destination, include_evaluation):
    source_pack, destination = Path(source_pack), Path(destination)
    source_manifest = read_json(source_pack / "source_manifest.json")
    manifest = read_json(source_pack / "pack_manifest.json")

    core = destination / "PACK-CORE"
    observations = core / "observations"
    observations.mkdir(parents=True)
    first_part = sorted(
        (source_pack / "PACK-CORE" / "observations")
        .glob("part-*.parquet")
    )[0]
    pd.read_parquet(first_part).head(200).to_parquet(
        observations / "part-00000.parquet", index=False
    )
    for table in (
        "metric_catalogue",
        "entity_registry",
        "entity_relations",
    ):
        shutil.copy2(
            source_pack / "PACK-CORE" / f"{table}.parquet",
            core / f"{table}.parquet",
        )

    evaluation_tables = []
    if include_evaluation and manifest["evaluation_tables"]:
        evaluation = destination / "PACK-EVAL"
        evaluation.mkdir()
        evaluation_tables = manifest["evaluation_tables"]
        for table in evaluation_tables:
            shutil.copy2(
                source_pack / "PACK-EVAL" / f"{table}.parquet",
                evaluation / f"{table}.parquet",
            )

    finalise_pack(
        destination,
        sector=manifest["sector"],
        pack_version=manifest["pack_version"],
        expected_cadence_seconds=manifest[
            "expected_cadence_seconds"
        ],
        source_manifest=source_manifest,
        evaluation_tables=evaluation_tables,
        notes=["bounded common-adapter isolation fixture"],
    )


def deliberately_leaky_scorer(run_root):
    evaluation_manifest = (
        Path(run_root) / "SPEC-EVAL" / "manifest.json"
    )
    if not evaluation_manifest.is_file():
        raise FileNotFoundError("SPEC-EVAL is not mounted")
    return sum(
        read_json(evaluation_manifest)["row_counts"].values()
    )


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    mounted_pack = temporary / "pack_with_truth"
    redacted_pack = temporary / "pack_without_truth"
    make_adapter_fixture(PACK_ROOT, mounted_pack, True)
    make_adapter_fixture(PACK_ROOT, redacted_pack, False)

    mounted_run = temporary / "run_with_truth"
    redacted_run = temporary / "run_without_truth"
    materialise_canonical(mounted_pack, mounted_run, True)
    materialise_canonical(redacted_pack, redacted_run, False)

    assert (
        core_content_hashes(mounted_run / "SPEC-CORE")
        == core_content_hashes(redacted_run / "SPEC-CORE")
    )
    assert (
        runtime_probe(mounted_run / "SPEC-CORE")
        == runtime_probe(redacted_run / "SPEC-CORE")
    )

    negative_control_detected = False
    try:
        deliberately_leaky_scorer(redacted_run)
    except FileNotFoundError:
        negative_control_detected = True
    assert negative_control_detected

print("PASS — SPEC-CORE is identical with and without PACK-EVAL")
print("PASS — the SPEC-CORE-only runtime output is identical")
print("PASS — the deliberately leaky scorer fails without SPEC-EVAL")


## 5. Acceptance report and next stage

Week 1 is complete for this sector when the pack-level isolation test and this
adapter-level isolation test both pass.

The next notebook should be **Canonical EDA**. It should read only
`SPEC-CORE`, inspect sampling, missingness, distributions, rolling behaviour,
ACF, and decomposition evidence, and must not use `SPEC-EVAL`. Feature
engineering and modelling follow only after that evidence is frozen.


In [ ]:
acceptance = {
    "contract_version": CORE_VERSION,
    "sector": pack_manifest["sector"],
    "pack_root": str(PACK_ROOT),
    "canonical_run_root": str(RUN_ROOT),
    "runtime_probe_succeeds_without_spec_eval": True,
    "runtime_test_scope": (
        "deployment evidence; pack and adapter invariance are primary"
    ),
    "core_equal_with_and_without_truth": True,
    "negative_control_detected": True,
    "adapter_has_sector_branch": False,
    "core_tables": list(CORE_SCHEMAS),
    "evaluation_tables": pack_manifest["evaluation_tables"],
    "canonical_content_hashes": core_content_hashes(CORE),
    "next_stage": "Canonical EDA using SPEC-CORE only",
}
acceptance_path = RUN_ROOT / "acceptance_report.json"
if not acceptance_path.exists():
    write_json(acceptance_path, acceptance)

display(pd.Series(acceptance, name="result").to_frame())
print("Saved:", acceptance_path)


## 6. Inspect every generated output

This is a **contract QA** section. It inventories every output file, prints
every JSON manifest/report, and previews every Parquet table from both the
sector pack and the canonical run.

Partitioned observations and telemetry can contain millions of rows, so they
are inspected through their complete file inventory, schema, total row count,
and first/last samples. Small tables are displayed in full; larger tables use
a bounded head/tail preview.

This section intentionally opens `SPEC-EVAL` for Week 1 verification. The
future canonical EDA notebook must read `SPEC-CORE` only.


In [ ]:
def inventory_files(root, stage):
    rows = []
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file():
            continue
        record = {
            "stage": stage,
            "file": str(path.relative_to(root)),
            "size_mb": round(path.stat().st_size / 1024**2, 3),
            "rows": pd.NA,
            "columns": pd.NA,
        }
        if path.suffix == ".parquet":
            parquet = pq.ParquetFile(path)
            record["rows"] = parquet.metadata.num_rows
            record["columns"] = len(parquet.schema_arrow.names)
        rows.append(record)
    return rows


inventory = pd.DataFrame(
    inventory_files(PACK_ROOT, "sector pack")
    + inventory_files(RUN_ROOT, "canonical run")
)
with pd.option_context("display.max_rows", None):
    display(inventory)

print(f"Pack files: {(inventory['stage'] == 'sector pack').sum():,}")
print(f"Canonical files: {(inventory['stage'] == 'canonical run').sum():,}")


In [ ]:
for stage, root in (
    ("SECTOR PACK", PACK_ROOT),
    ("CANONICAL RUN", RUN_ROOT),
):
    for path in sorted(Path(root).rglob("*.json")):
        relative = path.relative_to(root)
        print(f"\n{'=' * 80}\n{stage}: {relative}\n{'=' * 80}")
        print(json.dumps(read_json(path), indent=2, default=str))


In [ ]:
MAX_DISPLAY_ROWS = 50
SAMPLE_ROWS = 10


def display_schema(parquet):
    schema = parquet.schema_arrow
    display(pd.DataFrame({
        "column": schema.names,
        "dtype": [str(field.type) for field in schema],
    }))


def display_table(path, title):
    parquet = pq.ParquetFile(path)
    row_count = parquet.metadata.num_rows
    print(f"\n{'=' * 80}\n{title}\nRows: {row_count:,}")
    display_schema(parquet)
    frame = pd.read_parquet(path)
    if row_count <= MAX_DISPLAY_ROWS:
        display(frame)
    else:
        half = MAX_DISPLAY_ROWS // 2
        print(
            f"Showing first {half} and last {half} rows; "
            f"{row_count - MAX_DISPLAY_ROWS:,} rows omitted."
        )
        display(pd.concat([frame.head(half), frame.tail(half)]))


def display_partitioned(directory, title):
    parts = sorted(Path(directory).glob("part-*.parquet"))
    if not parts:
        print(f"\n{title}: no parts")
        return
    row_count = sum(
        pq.ParquetFile(path).metadata.num_rows for path in parts
    )
    print(
        f"\n{'=' * 80}\n{title}\n"
        f"Parts: {len(parts):,} | Rows: {row_count:,}"
    )
    display_schema(pq.ParquetFile(parts[0]))
    first = pd.read_parquet(parts[0]).head(SAMPLE_ROWS).copy()
    last = pd.read_parquet(parts[-1]).tail(SAMPLE_ROWS).copy()
    first.insert(0, "_sample", "first")
    last.insert(0, "_sample", "last")
    display(pd.concat([first, last], ignore_index=True))


In [ ]:
display_partitioned(
    PACK_ROOT / "PACK-CORE" / "observations",
    "PACK-CORE / observations",
)
display_partitioned(
    CORE / "telemetry",
    "SPEC-CORE / telemetry",
)

for stage, root in (
    ("SECTOR PACK", PACK_ROOT),
    ("CANONICAL RUN", RUN_ROOT),
):
    for path in sorted(Path(root).rglob("*.parquet")):
        if path.parent.name in {"observations", "telemetry"}:
            continue
        display_table(path, f"{stage}: {path.relative_to(root)}")

print("\nInspection complete.")
print("Canonical EDA input:", CORE)
print("Do not provide SPEC-EVAL to EDA or modelling code.")
